# Release evidence matrix


**What this is (instruction 64.40).** The evidence behind a release, gathered from the stored results rather than restated: per method, per organism, per scorecard level, and every gap still open. A row without evidence says so; nothing is filled in by assumption.

Built 2026-11-07.

In [1]:
import glob, json, os, re, pandas as pd
from parascape import atlas as AT, space_catalogue as SC
R = lambda *p: os.path.join(REPO, *p)

## 1 · Every method on each parasite

In [2]:
metrics = pd.read_csv(R('results/label_metrics_2026_11_07/overall.tsv'), sep='\t')
beats = metrics.assign(beats=metrics.accuracy_all > metrics.chance).groupby(['organism', 'strategy']).agg(
    labels=('target', 'size'), beats_chance_on=('beats', 'sum'), best_macro_f1=('macro_f1', 'max')).reset_index()
frames = []
for org in ('Tg', 'Pf'):
    m = pd.read_csv(R(f'results/inference_atlas_2026_11_07/manifest_{org}.tsv'), sep='\t')
    frames.append(m.merge(beats[beats.organism == org].drop(columns='organism'), on='strategy', how='left'))
methods = pd.concat(frames, ignore_index=True)
methods['evidence'] = methods.apply(lambda r: 'held-out per-gene rows + card' if r.heldout_genes > 0 else
                                    ('card (own held-out scorecard)' if r.card == 'measured' else 'none'), axis=1)
methods.to_csv(os.path.join(OUT, 'methods.tsv'), sep='\t', index=False)
methods.groupby(['organism', 'evidence']).size().unstack(fill_value=0)

evidence,card (own held-out scorecard),held-out per-gene rows + card
organism,,
Pf,25,14
Tg,25,14


In [3]:
methods[methods.heldout_genes > 0][['organism', 'strategy', 'labels', 'beats_chance_on', 'best_macro_f1', 'deployment_claims']].round(3)

,organism,strategy,labels,beats_chance_on,best_macro_f1,deployment_claims
6,Tg,feature_knn,9,4,0.471,7110
7,Tg,map_neighbours,9,1,0.47,0
8,Tg,cluster_guilt,9,0,0.105,0
10,Tg,layer_propagation,9,1,0.493,0
11,Tg,layer_vote,9,4,0.478,0
12,Tg,physical_partners,9,0,0.362,0
13,Tg,structural_homology,9,0,0.272,0
18,Tg,supervised_classifier,9,3,0.589,0
29,Tg,stratum_focus,9,4,0.471,0
30,Tg,triangulation,9,4,0.516,0


## 2 · Organism spaces

In [4]:
spaces = pd.DataFrame([dict(code=e.code, species=e.species, kind=e.kind, distribution=e.distribution, state=e.state,
                            calibrated=e.calibrated, note=e.note) for e in SC.catalogue()])
spaces.to_csv(os.path.join(OUT, 'spaces.tsv'), sep='\t', index=False)
spaces

,code,species,kind,distribution,state,calibrated,note
0,Tg,Toxoplasma gondii,parasite,wheel,ready,True,
1,Pf,Plasmodium falciparum,parasite,wheel,ready,True,
2,Hs,Homo sapiens,host,pack,installable,False,a verified pack is available in this checkout
3,Mm,Mus musculus,host,pack,installable,False,a verified pack is available in this checkout


## 3 · Scorecard levels and where each is stored

In [5]:
levels = pd.DataFrame([
  dict(level='gene', what='each gene hidden with its orthogroup: the call, right or wrong', where='parascape/data/track_record.parquet; HarnessResult.rows',
       present=os.path.exists(R('parascape/data/track_record.parquet'))),
  dict(level='class', what='recall / precision / F1 per class, confusion', where='results/label_metrics_2026_11_07/by_class.tsv',
       present=os.path.exists(R('results/label_metrics_2026_11_07/by_class.tsv'))),
  dict(level='label', what='coverage, accuracy against chance, macro F1', where='results/label_metrics_2026_11_07/overall.tsv',
       present=os.path.exists(R('results/label_metrics_2026_11_07/overall.tsv'))),
  dict(level='method', what='calibration scorecard per method and organism', where='parascape/data/strategy_calibration.json',
       present=os.path.exists(R('parascape/data/strategy_calibration.json'))),
  dict(level='meta', what='a model over methods, strictly nested, admitted per label', where='results/meta_inference_nested_2026_11_07*',
       present=bool(glob.glob(R('results/meta_inference_nested_2026_11_07*')))),
  dict(level='question', what='harness frozen tests, touched once and logged', where='results/harness_holdout_log.jsonl',
       present=os.path.exists(R('results/harness_holdout_log.jsonl')))])
levels.to_csv(os.path.join(OUT, 'levels.tsv'), sep='\t', index=False)
levels

,level,what,where,present
0,gene,"each gene hidden with its orthogroup: the call, right or wrong",parascape/data/track_record.parquet; HarnessResult.rows,True
1,class,"recall / precision / F1 per class, confusion",results/label_metrics_2026_11_07/by_class.tsv,True
2,label,"coverage, accuracy against chance, macro F1",results/label_metrics_2026_11_07/overall.tsv,True
3,method,calibration scorecard per method and organism,parascape/data/strategy_calibration.json,True
4,meta,"a model over methods, strictly nested, admitted per label",results/meta_inference_nested_2026_11_07*,True
5,question,"harness frozen tests, touched once and logged",results/harness_holdout_log.jsonl,True


## 4 · Host pilots, measured on host truth

In [6]:
hs = pd.read_csv(R('results/human_inference_pilot_2026_11_07/scores.tsv'), sep='\t').assign(organism='Hs')
mm = pd.read_csv(R('results/mouse_inference_pilot_2026_11_07/scores.tsv'), sep='\t')
mm = mm[mm.partition.str.startswith('frozen')].assign(organism='Mm', location='BMDM surface', frozen_auc=lambda d: d.roc_auc.round(3),
                                                       frozen_ap=lambda d: d.average_precision.round(3), may_fill=False)
hosts = pd.concat([hs[['organism', 'location', 'frozen_auc', 'frozen_ap', 'may_fill']], mm[['organism', 'location', 'frozen_auc', 'frozen_ap', 'may_fill']]])
hosts.to_csv(os.path.join(OUT, 'host_pilots.tsv'), sep='\t', index=False)
hosts

,organism,location,frozen_auc,frozen_ap,may_fill
0,Hs,Nucleoplasm,0.639,0.508,False
1,Hs,Plasma membrane,0.673,0.197,False
2,Hs,Mitochondria,0.709,0.268,False
3,Hs,Golgi apparatus,0.579,0.072,False
4,Hs,Endoplasmic reticulum,0.689,0.188,False
1,Mm,BMDM surface,0.798,0.279,False


## 5 · Every unresolved gap

Open progress items (from `instructions/INDEX.md` and the open 64.x items), and the gaps the finished 64.x items stated in their own write-ups.

In [7]:
index = open(R('instructions/INDEX.md')).read()
open_items = [dict(item=m.group(1), title=re.sub(r'\s+', ' ', m.group(3))[:120]) for m in
              re.finditer(r'^\| (\d+) \| \[([^\]]+)\]\(open/[^)]+\)(.*?)\|\s*$', index, flags=re.M)]
sub_open = [dict(item=os.path.basename(p)[:5].replace('_', '.'), title=open(p).readline().strip('# \n'))
            for p in sorted(glob.glob(R('instructions/open/64_*.md'))) if 'Status: DONE' not in open(p).read()]
stated = []
for p in sorted(glob.glob(R('instructions/done/64_*.md'))) + sorted(glob.glob(R('instructions/open/64_*.md'))):
    text = open(p).read()
    for m in re.finditer(r'Gaps, stated:(.*?)(?:\n\n|$)', text, flags=re.S):
        stated.append(dict(item=os.path.basename(p)[:5].replace('_', '.'), gap=re.sub(r'\s+', ' ', m.group(1)).strip()[:400]))
gaps = pd.DataFrame(open_items + sub_open)
pd.DataFrame(stated).to_csv(os.path.join(OUT, 'stated_gaps.tsv'), sep='\t', index=False)
gaps.to_csv(os.path.join(OUT, 'open_items.tsv'), sep='\t', index=False)
print(len(open_items), 'open top-level items;', len(sub_open), 'open 64.x items;', len(stated), 'stated gaps in 64.x write-ups')
pd.DataFrame(stated)

17 open top-level items; 3 open 64.x items; 7 stated gaps in 64.x write-ups


,item,gap
0,64.21,"Hs is not yet registered in `organisms.SPACES` (the app does not list it until 64.24 wires host spaces in), the pack is not hosted for download, and biological benchmarks on human labels belong to 64.28 (the human inference pack). No PyPI or pack release before the licence review in item 90."
1,64.22,"the source files (assembled 2026-07-17) do not record Bgee/Ensembl release versions and their assembling script is not in the tree; gene->protein comes from one source column, not reviewed many-to-many cross-references; there are no observed mouse labels yet, so benchmarks belong to 64.29; Mm is not registered in `organisms.SPACES` until 64.24 wires host spaces into the app."
2,64.23,"a host space does not open as a full map window (the window is built for parasite tables; a host map window would be a separate item); packs are found in this checkout, not yet hosted for download; other instruction-53 species have no builder yet and are not listed."
3,64.24,"the 18 curated-literature host targets do not record a host species, so none opens until the source says; the human and mouse packs are built but not hosted for download, so a fresh install shows ""not installed"" until a pack is installed (`python -m parascape.packs install …`); inference through a bridge still needs its own registered benchmark."
4,64.25,partitions run serially in one process (parallel workers remain `build_track_record.py --workers`); code identity is coarse; deployment outputs (calls on unknown genes) are not yet partitioned this way.
5,64.28,"** no human homology groups (paralogs can sit on both sides of a split); only expression evidence in the pack (no human interaction, structure or domain columns yet); HPA locations are cell-line immunofluorescence; catalogue strategies other than the harness models need a human context adapter (not built), so human has no per-gene track record and stays ""not calibrated"" in the space catalogue."
6,64.29,"** one observed mouse label in the pilot; no mouse homology groups (paralogs can sit on both sides of a split); BMDM surface proteomics measures one cultured cell type; catalogue strategies other than the harness models need a mouse context adapter (not built), so mouse has no per-gene track record and stays ""not calibrated"" in the space catalogue."
